# 🎬 Movio Platform — Automated Cloud Ingestion Pipeline (Google Colab)
**Production Cloud Pipeline for Headless WordPress (Pantheon) & Next.js**

- **TMDB API**: High-resolution official metadata, poster, backdrop, rating, and synopsis.
- **YTS API**: Best-quality torrent acquisition (1080p / 720p / magnet).
- **Aria2c**: Multi-connection cloud download with pre-sanitization (clean `.mp4` container).
- **Arabic-Only Subtitle Engine**: Searches and downloads ONLY the highest-rated Arabic (`.srt`) subtitle file via YIFY/IMDb.
- **FFmpeg Hardsubbing (Burn-in)**: Burns Arabic subtitles directly into `.mp4` video frames with crisp outlined typography (`-c:a copy` for lossless audio).
- **Resilient Upload Engine**: Auto-retry with fresh server allocation if large video uploads drop.
- **Clean DoodStream Embeds**: Direct video streaming embeds without query parameters.
- **Pantheon REST API**: Headless WordPress post publishing with strictly double-quoted 16:9 responsive embed iframe and custom metadata.

In [ ]:
# @title ⚙️ 1. Install Dependencies & Cloud Environment
!apt-get update -qq
!apt-get install -y -qq aria2 ffmpeg
!pip install -q requests requests-toolbelt beautifulsoup4 python-dotenv tqdm
!mkdir -p /content/download
print("✅ Cloud Environment Ready: aria2c, FFmpeg, and Python dependencies installed.")

In [ ]:
# @title 🔑 2. Cloud Configuration & Credentials (DevSecOps)
# @markdown Configure your target Pantheon WordPress site and API keys.
# @markdown Secrets can also be stored securely in Colab's encrypted Secrets sidebar.

WP_SITE_URL = "https://dev-movio-stream.pantheonsite.io" # @param {type:"string"}
WP_USERNAME = "admin" # @param {type:"string"}
WP_APP_PASSWORD = "" # @param {type:"string"}
TMDB_API_KEY = "" # @param {type:"string"}
DOODSTREAM_API_KEY = "" # @param {type:"string"}

# DevSecOps: Fallback to Colab Encrypted Secrets (key icon in sidebar)
try:
    from google.colab import userdata
    WP_APP_PASSWORD = WP_APP_PASSWORD or userdata.get('WP_APP_PASSWORD')
    TMDB_API_KEY = TMDB_API_KEY or userdata.get('TMDB_API_KEY')
    DOODSTREAM_API_KEY = DOODSTREAM_API_KEY or userdata.get('DOODSTREAM_API_KEY')
except Exception:
    pass

# DevSecOps: Fallback to Environment Variables
import os
WP_APP_PASSWORD = WP_APP_PASSWORD or os.getenv('WP_APP_PASSWORD', '')
TMDB_API_KEY = TMDB_API_KEY or os.getenv('TMDB_API_KEY', '')
DOODSTREAM_API_KEY = DOODSTREAM_API_KEY or os.getenv('DOODSTREAM_API_KEY', '')

WP_SITE_URL = WP_SITE_URL.rstrip('/')
print(f"✅ Target WordPress CMS: {WP_SITE_URL}")
print(f"✅ Credential Status: WP Auth: {'Configured' if WP_APP_PASSWORD else 'Missing'}, TMDB: {'Configured' if TMDB_API_KEY else 'Missing'}, DoodStream: {'Configured' if DOODSTREAM_API_KEY else 'Missing'}")

In [ ]:
# @title 🌐 3. TMDB Metadata & YTS Torrent Engine
import os, sys, re, time, json, shutil, zipfile, io, requests, subprocess
from urllib.parse import quote
from requests.auth import HTTPBasicAuth
from bs4 import BeautifulSoup
from requests_toolbelt import MultipartEncoder, MultipartEncoderMonitor

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Accept": "application/json, text/plain, */*",
}

def fetch_tmdb_metadata(title, year=None, imdb_id=None):
    print(f"[TMDB] Querying metadata for '{title}' (Year: {year or 'Any'}, IMDb: {imdb_id or 'None'})...")
    item = None
    if imdb_id and TMDB_API_KEY:
        try:
            res = requests.get(f"https://api.themoviedb.org/3/find/{imdb_id}?api_key={TMDB_API_KEY}&external_source=imdb_id", headers=HEADERS, timeout=15).json()
            results = res.get("movie_results", [])
            if results: item = results[0]
        except Exception as e: print(f"[TMDB] IMDb lookup error: {e}")

    if not item and TMDB_API_KEY:
        try:
            url = f"https://api.themoviedb.org/3/search/movie?api_key={TMDB_API_KEY}&query={quote(title)}"
            if year: url += f"&year={year}"
            res = requests.get(url, headers=HEADERS, timeout=15).json()
            results = res.get("results", [])
            if results: item = results[0]
        except Exception as e: print(f"[TMDB] Search error: {e}")

    genres = "Action, Drama"
    rating = "7.5"
    overview = "No synopsis available."
    poster_url = None
    backdrop_url = None
    final_title = title
    final_year = year or "2026"
    extracted_imdb = imdb_id

    if item:
        final_title = item.get("title") or title
        final_year = (item.get("release_date") or "")[:4] or final_year
        overview = item.get("overview") or overview
        rating = str(round(item.get("vote_average", 7.5), 1))
        if item.get("poster_path"): poster_url = f"https://image.tmdb.org/t/p/original{item['poster_path']}"
        if item.get("backdrop_path"): backdrop_url = f"https://image.tmdb.org/t/p/original{item['backdrop_path']}"
        if item.get("id") and TMDB_API_KEY:
            try:
                d = requests.get(f"https://api.themoviedb.org/3/movie/{item['id']}?api_key={TMDB_API_KEY}", headers=HEADERS, timeout=15).json()
                if "genres" in d: genres = ", ".join([g["name"] for g in d["genres"]])
                extracted_imdb = d.get("imdb_id") or extracted_imdb
            except Exception: pass

    if not poster_url:
        poster_url = "https://images.unsplash.com/photo-1489599849927-2ee91cede3ba?w=1000&q=85"

    meta = {
        "title": final_title, "year": final_year, "overview": overview,
        "rating": rating, "genres": genres, "poster_url": poster_url,
        "backdrop_url": backdrop_url, "imdb_id": extracted_imdb
    }
    print(f"[TMDB] ✅ Found: '{meta['title']}' ({meta['year']}) | ★ {meta['rating']} | {meta['genres']} | IMDb: {meta['imdb_id']}")
    return meta

def fetch_yts_torrent(title, year=None, imdb_id=None, quality="1080p"):
    query = imdb_id if imdb_id else title
    print(f"[YTS] Searching torrent for '{query}'...")
    mirrors = ["https://yts.mx", "https://yts.nz", "https://yts.lt", "https://yts.do"]
    data = None
    for m in mirrors:
        try:
            r = requests.get(f"{m}/api/v2/list_movies.json?query_term={quote(query)}&sort_by=download_count", headers=HEADERS, timeout=15)
            if r.status_code == 200:
                rj = r.json()
                if rj.get("status") == "ok" and rj.get("data", {}).get("movie_count", 0) > 0:
                    data = rj["data"]
                    print(f"[YTS] Connected to {m} (Matches: {data['movie_count']})")
                    break
        except Exception: continue

    if not data or not data.get("movies"):
        raise RuntimeError(f"No torrents found on YTS for '{query}'.")

    movies = data["movies"]
    target = movies[0]
    if year:
        for mv in movies:
            if str(mv.get("year")) == str(year):
                target = mv; break

    torrents = target.get("torrents", [])
    if not torrents:
        raise RuntimeError(f"No torrent streams available for '{target.get('title')}'.")

    sel = None
    for q in [quality, "1080p", "720p", "2160p"]:
        for t in torrents:
            if t.get("quality", "").lower() == q.lower():
                sel = t; break
        if sel: break
    if not sel: sel = torrents[0]

    h = sel.get("hash")
    q = sel.get("quality", "1080p")
    trackers = ["udp://open.demonii.com:1337/announce", "udp://tracker.openbittorrent.com:80", "udp://tracker.opentrackr.org:1337/announce"]
    tr_args = "&".join([f"tr={quote(t)}" for t in trackers])
    magnet = f"magnet:?xt=urn:btih:{h}&dn={quote(target.get('title'))}&{tr_args}"
    print(f"[YTS] ✅ Selected: {q} ({sel.get('size')}) Hash: {h[:10]}...")
    return {"quality": q, "torrent_url": sel.get("url"), "magnet_uri": magnet, "size": sel.get("size")}

In [ ]:
# @title 📥 4. Aria2c Download & Sanitization Engine
def sanitize_download_dir(download_dir="/content/download"):
    """Clear incomplete chunks and control files to resolve aria2 exit status 13."""
    if not os.path.exists(download_dir):
        os.makedirs(download_dir, exist_ok=True)
        return
    print(f"[ARIA2] Sanitizing download directory: {download_dir}...")
    stale_exts = (".aria2", ".part", ".tmp", ".crdownload")
    for root, _, files in os.walk(download_dir):
        for f in files:
            if f.lower().endswith(stale_exts):
                try:
                    os.remove(os.path.join(root, f))
                    print(f"[ARIA2] Cleared stale file: {f}")
                except Exception: pass

def download_with_aria2(source_uri, download_dir="/content/download"):
    """High-speed cloud download using aria2c with overwrite and auto-renaming safety flags (keeps .mp4)."""
    sanitize_download_dir(download_dir)
    print(f"[ARIA2] Starting multi-connection download into {download_dir}...")
    cmd = [
        "aria2c", f"--dir={download_dir}", "--max-connection-per-server=16",
        "--split=16", "--min-split-size=1M", "--summary-interval=10",
        "--seed-time=0", "--follow-torrent=mem",
        "--allow-overwrite=true", "--auto-file-renaming=false",
        "--conditional-get=true", "--file-allocation=none",
        source_uri
    ]
    proc = subprocess.run(cmd, capture_output=False)
    if proc.returncode != 0:
        raise RuntimeError(f"aria2c download failed with exit code {proc.returncode}")

    videos = []
    for root, _, files in os.walk(download_dir):
        for f in files:
            if f.lower().endswith((".mp4", ".mkv", ".avi", ".webm")):
                p = os.path.join(root, f)
                videos.append((p, os.path.getsize(p)))

    if not videos:
        raise FileNotFoundError(f"No video file downloaded in {download_dir}.")
    videos.sort(key=lambda x: x[1], reverse=True)
    target_file = videos[0][0]
    print(f"[ARIA2] ✅ Downloaded: {os.path.basename(target_file)} ({videos[0][1]/(1024*1024):.1f} MB)")
    return target_file

In [ ]:
# @title 💬 5. Arabic Subtitles & FFmpeg Hardsubbing (Burn-in)
def download_subtitles_for_imdb(imdb_id, output_dir="/content/download"):
    """Search and download ONLY the highest-rated Arabic (.srt) subtitle file."""
    if not imdb_id:
        print("[SUBS] No IMDb ID available; skipping Arabic subtitle download.")
        return None

    subs_dir = os.path.join(output_dir, "subtitles")
    os.makedirs(subs_dir, exist_ok=True)
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"}
    print(f"[SUBS] Searching highest-rated Arabic subtitle for IMDb ID '{imdb_id}'...")
    arabic_sub_url = None

    # 1. Try JSON endpoint
    try:
        r = requests.get(f"https://api.yifysubtitles.ch/subs/{imdb_id}", headers=headers, timeout=6)
        if r.status_code == 200:
            data = r.json().get("subs", {}).get(imdb_id, {})
            arabic_subs = data.get("arabic", [])
            if arabic_subs:
                sorted_subs = sorted(arabic_subs, key=lambda x: x.get("rating", 0), reverse=True)
                arabic_sub_url = sorted_subs[0].get("url")
                if arabic_sub_url:
                    print(f"[SUBS] Found Arabic subtitle via JSON API (Rating: {sorted_subs[0].get('rating', 'N/A')})")
    except Exception as e:
        print(f"[SUBS] JSON API notice: {e}")

    # 2. Resilient fallback to HTML mirrors
    if not arabic_sub_url:
        mirrors = [
            f"https://yifysubtitles.ch/movie-imdb/{imdb_id}",
            f"https://yts-subs.com/movie-imdb/{imdb_id}",
            f"https://yifysubtitles.org/movie-imdb/{imdb_id}"
        ]
        for mirror in mirrors:
            try:
                r = requests.get(mirror, headers=headers, timeout=10)
                if r.status_code == 200:
                    soup = BeautifulSoup(r.text, "html.parser")
                    for tr in soup.find_all("tr"):
                        tag = tr.find(class_="sub-lang")
                        if not tag: continue
                        if tag.text.strip().lower() == "arabic":
                            a = tr.find("a", href=True)
                            if a and "/subtitles/" in a["href"]:
                                arabic_sub_url = a["href"]
                                print(f"[SUBS] Found Arabic subtitle on mirror: {mirror}")
                                break
                    if arabic_sub_url: break
            except Exception: continue

    if not arabic_sub_url:
        print(f"[SUBS] No Arabic subtitles found for IMDb ID {imdb_id}.")
        return None

    # 3. Download .zip archive and extract Arabic .srt
    try:
        slug = arabic_sub_url.rstrip("/").split("/")[-1]
        zip_url = f"https://yifysubtitles.ch/subtitle/{slug}.zip"
        req_headers = {"User-Agent": headers["User-Agent"], "Referer": f"https://yifysubtitles.ch/subtitles/{slug}"}
        zr = requests.get(zip_url, headers=req_headers, timeout=15)
        if zr.status_code == 200:
            with zipfile.ZipFile(io.BytesIO(zr.content)) as z:
                for filename in z.namelist():
                    if filename.lower().endswith(".srt"):
                        out_srt = os.path.join(subs_dir, f"{imdb_id}_ara.srt")
                        with open(out_srt, "wb") as sf:
                            sf.write(z.read(filename))
                        print(f"[SUBS] ✅ Extracted Arabic subtitle -> {os.path.basename(out_srt)}")
                        return out_srt
    except Exception as e:
        print(f"[SUBS] Failed downloading/extracting Arabic subtitle: {e}")

    return None

def burn_arabic_subtitles(video_path, srt_path):
    """Burn Arabic subtitles directly into video frames (hardsubbing) using FFmpeg."""
    if not srt_path or not os.path.exists(srt_path):
        print("[HARDSUB] ⚠️ No Arabic subtitle provided or file missing; using original video as fallback.")
        return video_path

    ffmpeg_bin = shutil.which("ffmpeg")
    if not ffmpeg_bin:
        print("[HARDSUB] ⚠️ ffmpeg not found in PATH; skipping hardsubbing and using raw video.")
        return video_path

    base_name, _ = os.path.splitext(video_path)
    output_subbed_path = f"{base_name}_subbed.mp4"
    if os.path.abspath(output_subbed_path) == os.path.abspath(video_path):
        output_subbed_path = f"{base_name}_burned.mp4"

    # Escape special characters in the subtitle file path for FFmpeg filter syntax (escape ':' and ''')
    clean_path = os.path.abspath(srt_path).replace("\\", "/")
    escaped_srt = clean_path.replace(":", "\\:").replace("'", "\\'")

    # Clear, readable Arabic styling: White text, black outline, shadow
    style = "FontSize=22,PrimaryColour=&H00FFFFFF,OutlineColour=&H00000000,BorderStyle=1,Outline=2,Shadow=1"
    subtitles_filter = f"subtitles='{escaped_srt}':force_style='{style}'"

    print(f"[HARDSUB] Burning Arabic subtitles into frames: {os.path.basename(output_subbed_path)}...")
    cmd = [
        ffmpeg_bin, "-y",
        "-i", video_path,
        "-vf", subtitles_filter,
        "-c:v", "libx264",
        "-preset", "veryfast",
        "-crf", "22",
        "-c:a", "copy",
        output_subbed_path
    ]
    proc = subprocess.run(cmd, capture_output=True, text=True)
    if proc.returncode != 0:
        print(f"[HARDSUB] ❌ ffmpeg error ({proc.returncode}): {proc.stderr[-300:]}")
        print("[HARDSUB] Falling back to original video file.")
        return video_path

    if os.path.exists(output_subbed_path) and os.path.getsize(output_subbed_path) > 0:
        mb = os.path.getsize(output_subbed_path) / (1024 * 1024)
        print(f"[HARDSUB] ✅ Hardsubbing complete! Video: {os.path.basename(output_subbed_path)} ({mb:.1f} MB)")
        return output_subbed_path

    return video_path

In [ ]:
# @title ☁️ 6. Resilient DoodStream Upload Engine
def upload_to_doodstream(video_path, max_retries=3):
    """Chunked streaming upload to DoodStream API with retry loop and new server allocation on disconnect."""
    if not DOODSTREAM_API_KEY:
        raise ValueError("DOODSTREAM_API_KEY is not configured. Please supply an API key in Cell 2.")

    fname = os.path.basename(video_path)
    size_mb = os.path.getsize(video_path) / (1024 * 1024)
    last_err = None

    for attempt in range(1, max_retries + 1):
        try:
            print(f"[DOOD] Requesting fresh upload server (Attempt {attempt}/{max_retries})...")
            srv_resp = requests.get("https://doodapi.com/api/upload/server", params={"key": DOODSTREAM_API_KEY}, timeout=30).json()
            if srv_resp.get("status") != 200 or not srv_resp.get("result"):
                raise RuntimeError(f"Failed to allocate upload server: {srv_resp}")

            upload_url = srv_resp["result"]
            print(f"[DOOD] Uploading '{fname}' ({size_mb:.1f} MB) via {upload_url[:45]}...")

            with open(video_path, 'rb') as f:
                encoder = MultipartEncoder(fields={'api_key': DOODSTREAM_API_KEY, 'file': (fname, f, 'video/mp4')})
                last_pct = [-1]
                def progress(m):
                    pct = int((m.bytes_read / m.len) * 100)
                    if pct % 10 == 0 and pct != last_pct[0]:
                        last_pct[0] = pct
                        print(f"  [Upload Progress] {pct}% ({m.bytes_read/(1024*1024):.1f} MB / {m.len/(1024*1024):.1f} MB)", flush=True)
                monitor = MultipartEncoderMonitor(encoder, progress)
                resp = requests.post(upload_url, data=monitor, headers={'Content-Type': monitor.content_type}, timeout=7200).json()

            if resp.get("status") != 200:
                raise RuntimeError(f"DoodStream rejected upload: {resp}")

            res = resp["result"]
            code = res[0]["filecode"] if isinstance(res, list) else res["filecode"]
            embed = f"https://doodstream.com/e/{code}"
            print(f"[DOOD] ✅ Upload successful! Embed: {embed}")
            return embed
        except Exception as e:
            last_err = e
            print(f"[DOOD] Upload attempt {attempt} notice: {e.__class__.__name__}: {e}")
            if attempt < max_retries:
                wait_sec = attempt * 7
                print(f"[DOOD] Waiting {wait_sec}s before requesting a fresh upload server...")
                time.sleep(wait_sec)

    raise RuntimeError(f"DoodStream upload failed after {max_retries} attempts. Last error: {last_err}")

In [ ]:
# @title 📝 7. Headless WordPress Publisher (Strict Double Quotes)
def upload_poster_to_pantheon(image_url, slug):
    if not image_url: return None
    print(f"[WP] Downloading TMDB poster: {image_url[:80]}...")
    img_data = requests.get(image_url, headers=HEADERS, timeout=30).content
    h = {
        "Content-Disposition": f'attachment; filename="{slug[:25]}_{int(time.time())}.jpg"',
        "Content-Type": "image/jpeg",
        "User-Agent": HEADERS["User-Agent"]
    }
    auth = HTTPBasicAuth(WP_USERNAME, WP_APP_PASSWORD) if WP_APP_PASSWORD else None
    res = requests.post(f"{WP_SITE_URL}/wp-json/wp/v2/media", headers=h, data=img_data, auth=auth, timeout=40)
    if res.status_code in (200, 201):
        mid = res.json()["id"]
        print(f"[WP] ✅ Poster attached to Pantheon Media Library (ID: {mid})")
        return mid
    print(f"[WP] Poster upload notice ({res.status_code}): {res.text[:100]}")
    return None

def publish_movie_to_pantheon(meta, embed_url, quality="1080p"):
    """Publish post to Pantheon WordPress with strict double quotes on iframe for Next.js regex parser."""
    slug = re.sub(r'[^a-zA-Z0-9]+', '-', meta['title'].lower()).strip('-')
    media_id = upload_poster_to_pantheon(meta.get("poster_url"), slug)
    title = f"{meta['title']} ({meta['year']})"
    print(f"[WP] Publishing '{title}' to Pantheon WordPress...")

    # STRICT double quotes on iframe attributes ensure Next.js player matches cleanly
    content = f"""<div class="video-container" style="position: relative; padding-bottom: 56.25%; height: 0; overflow: hidden; max-width: 100%; border-radius: 12px; margin-bottom: 1.5rem;">
    <iframe src="{embed_url}" style="position: absolute; top: 0; left: 0; width: 100%; height: 100%; border: 0;" allowfullscreen="true" scrolling="no" frameborder="0"></iframe>
</div>
<div class="movie-meta-summary">
    <p><strong>Rating:</strong> {meta.get('rating', '8.0')} / 10</p>
    <p><strong>Release Year:</strong> {meta.get('year', '2026')}</p>
    <p><strong>Genres:</strong> {meta.get('genres', 'Movies')}</p>
    <p><strong>Quality:</strong> {quality} Full HD</p>
    <hr>
    <p>{meta.get('overview', '')}</p>
</div>"""

    payload = {
        "title": title,
        "content": content,
        "status": "publish",
        "meta": {
            "embed_url": embed_url,
            "dood_embed": embed_url,
            "video_year": str(meta.get("year", "2026")),
            "imdb_rating": str(meta.get("rating", "8.0")),
            "quality": quality,
            "backdrop_url": meta.get("backdrop_url") or "",
            "genres": meta.get("genres", "")
        }
    }
    if media_id: payload["featured_media"] = media_id
    auth = HTTPBasicAuth(WP_USERNAME, WP_APP_PASSWORD) if WP_APP_PASSWORD else None
    res = requests.post(f"{WP_SITE_URL}/wp-json/wp/v2/posts", json=payload, headers=HEADERS, auth=auth, timeout=30)
    if res.status_code in (200, 201):
        data = res.json()
        print(f"🎉 SUCCESS: Movie published to Pantheon WordPress!")
        print(f"Post ID:   {data.get('id')}")
        print(f"Post Slug: {data.get('slug')}")
        print(f"Live Post: {data.get('link')}")
        return data
    raise RuntimeError(f"WP publish failed ({res.status_code}): {res.text[:250]}")

In [ ]:
# @title 🚀 8. Master Pipeline Execution
# @markdown Specify Movie Name and Year (or IMDb ID):

MOVIE_TITLE = "The Beekeeper" # @param {type:"string"}
RELEASE_YEAR = "2024" # @param {type:"string"}
IMDB_ID = "" # @param {type:"string"}
PREFERRED_QUALITY = "1080p" # @param ["1080p", "720p", "2160p"]

def run_pipeline(title, year=None, imdb=None, quality="1080p"):
    print("=" * 75)
    print("  MOVIO CLOUD AUTOMATION PIPELINE (TMDB + YTS + ARIA2C + ARABIC HARDSUB + PANTHEON)")
    print("=" * 75)
    # 1. Acquire TMDB metadata & YTS torrent
    meta = fetch_tmdb_metadata(title.strip(), year.strip() if year else None, imdb.strip() if imdb else None)
    target_imdb = meta.get("imdb_id") or (imdb.strip() if imdb else None)
    torrent = fetch_yts_torrent(meta["title"], meta["year"], target_imdb, quality)

    # 2. Download movie .mp4 using Aria2c into /content/download/
    raw_video = download_with_aria2(torrent["torrent_url"] or torrent["magnet_uri"])

    # 3. Fetch Arabic .srt via download_subtitles_for_imdb
    arabic_srt = download_subtitles_for_imdb(target_imdb, "/content/download")

    # 4. Burn Arabic subtitles using burn_arabic_subtitles -> *_subbed.mp4
    burned_video = burn_arabic_subtitles(raw_video, arabic_srt)

    # 5. Upload burned *_subbed.mp4 to DoodStream
    embed_url = upload_to_doodstream(burned_video)

    # 6. Publish directly to Pantheon WordPress with clean DoodStream embed URL
    post_data = publish_movie_to_pantheon(meta, embed_url, torrent["quality"])

    # 7. Clean up temporary files: original .mp4, .srt, and generated *_subbed.mp4
    cleanup_list = set([raw_video, burned_video, arabic_srt])
    for p in cleanup_list:
        if p and os.path.exists(p):
            try:
                os.remove(p)
                print(f"[CLEANUP] Deleted temporary file: {os.path.basename(p)}")
            except Exception: pass

    print("\n" + "=" * 75)
    print("  PIPELINE COMPLETED SUCCESSFULLY!")
    print(f"  Movie:      {meta['title']} ({meta['year']})")
    print(f"  Rating:     ★ {meta['rating']}")
    print(f"  Arabic Sub: {'Burned In' if arabic_srt else 'None'}")
    print(f"  Embed URL:  {embed_url}")
    print(f"  Live Post:  {post_data.get('link')}")
    print(f"  Next.js:    https://dev-movio-stream.pantheonsite.io/movie/{post_data.get('slug')}")
    print("=" * 75)
    return post_data

# Execute pipeline with configured inputs
post_result = run_pipeline(MOVIE_TITLE, RELEASE_YEAR, IMDB_ID, PREFERRED_QUALITY)